# Sasol Customer Value & Opportunity Prediction
## In-Depth Exploratory Data Analysis & Model Architecture Insights

This notebook presents an in-depth empirical exploration of point-of-sale receipt streams, target distributions, zero-inflation dynamics, and multi-class transition behaviors. It synthesizes visual findings into actionable modeling strategies for `CLV_fuel`, `CLV_nonfuel`, and the 17 `Opportunity` states.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from sasol_customer_value.config import (
    NORMALIZATION_CONSTANTS, OPPORTUNITY_LABELS, TARGET_FUEL, TARGET_NONFUEL, TARGET_OPPORTUNITY
)
from sasol_customer_value.data.loader import load_transactions

S_FUEL = NORMALIZATION_CONSTANTS[TARGET_FUEL]
S_NONFUEL = NORMALIZATION_CONSTANTS[TARGET_NONFUEL]
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Environment and libraries initialized.')

### 1. Target Distributions, Zero Inflation & Log Compression

The two continuous targets ($CLV_{fuel}$ and $CLV_{nonfuel}$) represent customer activity over a forward 3-month window:
* **$CLV_{fuel}$**: Log-normalized fuel litres ($s_{fuel} = 2.3528$). Exactly **29.2%** of eligible loyalty customers purchase zero litres in the forward 3-month window.
* **$CLV_{nonfuel}$**: Log-normalized convenience store spend ($s_{nonfuel} = 2.4201$). Exactly **65.1%** of eligible loyalty customers purchase zero store items in the forward 3-month window.

**Key Insight:** Log transformation $z = \ln(1 + y) / s$ eliminates extreme right-tail skewness while keeping the distribution continuous and well-behaved for tree regressors.

In [2]:
display(Image(filename='../reports/figures/fig1_target_distributions.png'))

### 2. Opportunity 17-Class Imbalance & Multi-Class Dynamics

The 17 mutually exclusive Opportunity classes exhibit massive class imbalance:
1. **Dominant Baseline States:**
   * `Stable`: ~32.4% (customers who continue buying but spend does not grow $\ge 20$ ZAR by $\ge 25\%$).
   * `Inactivity`: ~20.9% (customers who completely churn from qualifying items).
2. **Frequent Growth & Adoption Drivers:**
   * `Fuel` Growth: ~17.5%
   * `Food & beverage`: ~6.7%
   * `Beverages`: ~5.5%
3. **Long-Tail Categories:**
   * `Braai & Ice`, `Confectionary`, `Lubricants`, `Ice-cream` represent $< 1.5\%$ of outcomes each.

**Key Insight:** Standard `argmax` prediction severely suppresses minority categories, hurting Weighted F1. Multi-model ensembling and probability threshold calibration are essential.

In [3]:
display(Image(filename='../reports/figures/fig2_opportunity_class_imbalance.png'))

### 3. Macro Temporal Dynamics & Annual Seasonality

Tracking point-of-sale volume across the multi-year history reveals pronounced temporal cycles:
* **The Summer Road Trip Surge (December – February):** Significant uplift in fuel litres dispensed and convenience store basket sizes.
* **1-Year Seasonal Persistence:** Customers frequently repeat holiday habits or commuter routines from the corresponding season of the previous year.

**Key Insight:** Including 1-year lookback features (`same_season_prior_year_baskets`, `same_season_prior_year_fuel_litres`, `same_season_prior_year_nonfuel_spend`) directly aligns model predictions with these macro retail cycles.

In [4]:
display(Image(filename='../reports/figures/fig3_temporal_trends_and_seasonality.png'))

### 4. Joint Multi-Task Behavior: Fuel vs Convenience Forecourt

Cross-analyzing fuel and non-fuel spend shows four clear behavioral archetypes:
* **Dual Shoppers (32.8%):** Customers who purchase both fuel and Sasol Delight store merchandise.
* **Fuel-Only Shoppers (37.9%):** Forecourt-only customers who never step into the convenience store.
* **Convenience-Only Shoppers (2.1%):** Very rare foot-traffic customers who buy only store items.
* **Completely Inactive (27.2%):** Customers who lapse across both categories over the 3-month window.

**Key Insight:** Because 37.9% of customers are fuel-only, passing predicted fuel spend as an input into the non-fuel and opportunity models (multi-task cascading) allows the classifier to eliminate impossible cross-category transitions.

In [5]:
display(Image(filename='../reports/figures/fig4_fuel_vs_nonfuel_joint_dynamics.png'))

### 5. Test Cohort Predictions & Sanity Audit

Examining model predictions on the 5,488 test customers for the forward evaluation period (Dec 2025 – Feb 2026):
* Continuous distributions for `CLV_fuel` and `CLV_nonfuel` match the shape and variance of ground truth validation actuals without clipping or truncation artifacts.
* The predicted Opportunity distribution exhibits healthy representation across baseline and growth categories.

In [6]:
display(Image(filename='../reports/figures/fig5_test_predictions_diagnostic.png'))